# Notebook 02 — Feature Engineering
**Project:** Credit Card Customer Profiling & Segmentation  
**Goal:** Impute missing values, create 7 behavioural ratio features, detect outliers, and scale for clustering.

---
| Step | Action |
|------|--------|
| 1 | Impute missing values with median |
| 2 | Create 7 engineered ratio features |
| 3 | Outlier analysis (IQR method) |
| 4 | StandardScaler — normalise all features |
| 5 | Save processed data |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
df = load_raw_data()
df = df.drop(columns=['CUST_ID'])
print('Raw shape:', df.shape)

## 1. Impute Missing Values

In [ ]:
print('Before imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

for col in df.columns:
    if df[col].isnull().any():
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f'  {col}: filled with median = {med:.2f}')

print(f'\nAfter imputation — missing values: {df.isnull().sum().sum()}')

## 2. Feature Engineering

Seven new ratio features derived from domain knowledge:

| Feature | Formula | Captures |
|---|---|---|
| `PURCHASES_TO_LIMIT_RATIO` | PURCHASES / CREDIT_LIMIT | Credit utilisation for spending |
| `CASH_ADVANCE_RATIO` | CASH_ADVANCE / BALANCE | Cash reliance vs balance |
| `PAYMENT_TO_MINIMUM_RATIO` | PAYMENTS / MINIMUM_PAYMENTS | Debt repayment aggressiveness |
| `MONTHLY_AVG_PURCHASE` | PURCHASES / TENURE | Average monthly spend rate |
| `INSTALLMENT_TO_PURCHASE_RATIO` | INSTALLMENTS / PURCHASES | Preference for installment buying |
| `CASH_ADVANCE_TO_CREDIT_RATIO` | CASH_ADVANCE / CREDIT_LIMIT | Cash advance vs credit limit |
| `BALANCE_TO_CREDIT_RATIO` | BALANCE / CREDIT_LIMIT | Balance utilisation rate |

In [ ]:
df['PURCHASES_TO_LIMIT_RATIO']      = (df['PURCHASES'] / (df['CREDIT_LIMIT'] + 1)).round(4)
df['CASH_ADVANCE_RATIO']            = (df['CASH_ADVANCE'] / (df['BALANCE'] + 1)).round(4)
df['PAYMENT_TO_MINIMUM_RATIO']      = (df['PAYMENTS'] / (df['MINIMUM_PAYMENTS'] + 1)).round(4)
df['MONTHLY_AVG_PURCHASE']          = (df['PURCHASES'] / df['TENURE']).round(2)
df['INSTALLMENT_TO_PURCHASE_RATIO'] = (df['INSTALLMENTS_PURCHASES'] / (df['PURCHASES'] + 1)).round(4)
df['CASH_ADVANCE_TO_CREDIT_RATIO']  = (df['CASH_ADVANCE'] / (df['CREDIT_LIMIT'] + 1)).round(4)
df['BALANCE_TO_CREDIT_RATIO']       = (df['BALANCE'] / (df['CREDIT_LIMIT'] + 1)).round(4)

eng_cols = ['PURCHASES_TO_LIMIT_RATIO','CASH_ADVANCE_RATIO','PAYMENT_TO_MINIMUM_RATIO',
            'MONTHLY_AVG_PURCHASE','INSTALLMENT_TO_PURCHASE_RATIO',
            'CASH_ADVANCE_TO_CREDIT_RATIO','BALANCE_TO_CREDIT_RATIO']

print(f'Total features after engineering: {df.shape[1]}')
df[eng_cols].describe().round(3)

In [ ]:
palette = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0','#00BCD4','#795548']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, (ax, col) in enumerate(zip(axes, eng_cols)):
    ax.hist(df[col], bins=35, color=palette[i], edgecolor='white', alpha=0.85)
    ax.axvline(df[col].median(), color='black', lw=1.4, ls='--',
               label=f'Median: {df[col].median():.2f}')
    ax.set_title(col.replace('_',' ').title(), fontsize=8, fontweight='bold')
    ax.legend(fontsize=7)
for ax in axes[len(eng_cols):]:
    ax.set_visible(False)
plt.suptitle('Engineered Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Outlier Analysis

In [ ]:
monetary_cols = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT', 'PAYMENTS']
palette2 = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col, color in zip(axes, monetary_cols, palette2):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.55),
               medianprops=dict(color='black', linewidth=2),
               flierprops=dict(marker='.', markersize=2, alpha=0.3))
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    n_out = ((df[col] < Q1-1.5*(Q3-Q1)) | (df[col] > Q3+1.5*(Q3-Q1))).sum()
    ax.set_title(f'{col}\n({n_out} outliers)', fontsize=8, fontweight='bold')
    ax.set_ylabel('Value ($)')
plt.suptitle('Outlier Analysis — Retained (represent genuine high-value customers)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Scaling with StandardScaler
KMeans is distance-based — unscaled features with larger ranges dominate distance calculations. StandardScaler sets each feature to mean=0, std=1.

In [ ]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

print('Scaling verification (first 3 features):')
for col in df.columns[:3]:
    print(f'  {col}: mean={df_scaled[col].mean():.4f}, std={df_scaled[col].std():.4f}')
print(f'\nFinal shape: {df_scaled.shape[0]:,} rows x {df_scaled.shape[1]} features')

In [ ]:
# Save via src pipeline
from src.preprocessing import preprocess
from src.data_loader import load_raw_data as lrd
df_s, df_u = preprocess(lrd(), save=True)
print('Saved: data/processed/cc_clean_scaled.csv')
print('Saved: data/processed/cc_clean_unscaled.csv')

## Summary

| Step | Result |
|---|---|
| Imputation | 314 values filled (median) |
| Engineered features | 7 new ratio features added |
| Total features | 24 features for clustering |
| Outlier decision | Retained — real high-value segments |
| Scaling | StandardScaler (mean=0, std=1) |

> **Next:** `03_KMeans_Clustering.ipynb` — find optimal K, fit KMeans, and visualise segments.